# 01 — Environment Check
Run this **once** after setting up your environment.  
Every cell should print ✓. Fix anything that prints ✗ before proceeding.

**Order:** `env_check` → `data_download` → `eda`

## 1. Python version

In [9]:
import sys
v = sys.version_info
ok = v.major == 3 and v.minor >= 10
print(f"{'✓' if ok else '✗'}  Python {v.major}.{v.minor}.{v.micro}  (need ≥ 3.10)")

✓  Python 3.13.13  (need ≥ 3.10)


## 2. Core packages

In [10]:
# Install any missing ones with: pip install <package>
packages = [
    ("torch",        "torch"),
    ("numpy",        "numpy"),
    ("pandas",       "pandas"),
    ("sklearn",      "scikit-learn"),
    ("matplotlib",   "matplotlib"),
    ("seaborn",      "seaborn"),
    ("tqdm",         "tqdm"),
    ("requests",     "requests"),
    ("PIL",          "Pillow"),
    ("yaml",         "pyyaml"),
    ("spacy",        "spacy"),
    ("transformers", "transformers"),
    ("peft",         "peft"),
    ("datasets",     "datasets"),
]

failed = []
for import_name, pip_name in packages:
    try:
        mod = __import__(import_name)
        ver = getattr(mod, "__version__", "ok")
        print(f"  ✓  {pip_name:<22} {ver}")
    except ImportError:
        print(f"  ✗  {pip_name:<22} MISSING  →  pip install {pip_name}")
        failed.append(pip_name)

# open_clip separate — import name differs
try:
    import open_clip
    print(f"  ✓  open-clip-torch        {getattr(open_clip,'__version__','ok')}")
except ImportError:
    print(f"  ✗  open-clip-torch         MISSING  →  pip install open-clip-torch")
    failed.append("open-clip-torch")

print()
if failed:
    print(f"  ✗  Run: pip install {' '.join(failed)}")
else:
    print("  ✓  All core packages present.")

  ✓  torch                  2.6.0+cu124
  ✓  numpy                  2.4.4
  ✓  pandas                 3.0.2
  ✓  scikit-learn           1.8.0
  ✓  matplotlib             3.10.8
  ✓  seaborn                0.13.2
  ✓  tqdm                   4.67.3
  ✓  requests               2.33.1
  ✓  Pillow                 12.2.0
  ✓  pyyaml                 6.0.3
  ✓  spacy                  3.8.14
  ✓  transformers           5.5.3
  ✓  peft                   0.19.1
  ✓  datasets               4.8.4
  ✓  open-clip-torch        3.3.0

  ✓  All core packages present.


## 3. PyTorch device (CUDA / CPU)

In [11]:
import torch

print(f"  PyTorch  : {torch.__version__}")

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  ✓  CUDA — {gpu}  ({mem:.1f} GB VRAM)")
    DEVICE = "cuda"
else:
    print("  ⚠  No CUDA GPU found — CPU only.")
    print("     Fine for EDA and feature inspection.")
    print("     Training will need Colab/Kaggle GPU.")
    DEVICE = "cpu"

# Smoke test
t = torch.ones(3, 3).to(DEVICE)
print(f"  ✓  Tensor smoke test passed on {DEVICE}")
print(f"  Active device: {DEVICE}")

  PyTorch  : 2.6.0+cu124
  ✓  CUDA — NVIDIA GeForce RTX 4060 Ti  (8.6 GB VRAM)
  ✓  Tensor smoke test passed on cuda
  Active device: cuda


## 4. SpaCy NER model

In [12]:
import spacy

# You installed en_core_web_sm — that's what we use here.
# en_core_web_trf is more accurate but needs transformers extras.
# We'll use sm for EDA; trf is optional for Stream A fine-tuning.
for model in ["en_core_web_trf", "en_core_web_sm"]:
    try:
        nlp = spacy.load(model)
        doc = nlp("Barack Obama visited Paris on Monday.")
        ents = [(e.text, e.label_) for e in doc.ents]
        print(f"  ✓  {model} loaded")
        print(f"     Test entities : {ents}")
        SPACY_MODEL = model
        break
    except OSError:
        print(f"  ⚠  {model} not found — trying next...")
else:
    print("  ✗  No SpaCy model found.")
    print("     Run: python -m spacy download en_core_web_sm")
    SPACY_MODEL = None

  ⚠  en_core_web_trf not found — trying next...
  ✓  en_core_web_sm loaded
     Test entities : [('Barack Obama', 'PERSON'), ('Paris', 'GPE'), ('Monday', 'DATE')]


## 5. OpenCLIP — ViT-B/32 smoke test

In [13]:
# We do NOT load ViT-L/14 here — it's 900 MB and slow to download.
# This just checks open_clip is installed and can encode text.
try:
    import open_clip, torch
    model, _, preprocess = open_clip.create_model_and_transforms(
        "ViT-B-32", pretrained="openai"
    )
    tokenizer = open_clip.get_tokenizer("ViT-B-32")
    model.eval()
    with torch.no_grad():
        feats = model.encode_text(tokenizer(["a news photo"]))
    print(f"  ✓  OpenCLIP ViT-B/32 loaded")
    print(f"     Text embedding shape: {feats.shape}  (should be [1, 512])")
except ImportError:
    print("  ✗  open_clip not installed  →  pip install open-clip-torch")
except Exception as e:
    print(f"  ✗  OpenCLIP error: {e}")

open_clip_model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

C:\Users\User\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--timm--vit_base_patch32_clip_224.openai. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\User\AppData\Roaming\Python\Python313\site-packages\open_clip\factory.py:450: Us

  ✓  OpenCLIP ViT-B/32 loaded
     Text embedding shape: torch.Size([1, 512])  (should be [1, 512])


## 6. Pre-extracted embeddings — quick peek

In [14]:
import pickle, numpy as np
from pathlib import Path
import sys
sys.path.insert(0, str(Path("../../").resolve()))
from src.utils.paths import *

print("NewsCLIPpings CLIP embeddings (ViT-B/32):" )
for folder, split in [("clip_image_embeddings", "test"), ("clip_text_embeddings", "test")]:
    fpath = CLIP_EMB / folder / f"{split}.pkl"
    if fpath.exists():
        data = pickle.load(open(fpath, "rb"))
        sample = next(iter(data.values()))
        print(f"  ✓  {folder}/{split}.pkl — {len(data):,} entries  dim={sample.shape[0]}")
    else:
        print(f"  ✗  {fpath} not found")

print()
print("VERITE CLIP embeddings (ViT-L/14):")
for fname in ["image_embeddings.npy", "text_embeddings.npy"]:
    fpath = CLIP_EMB / "verite" / fname
    if fpath.exists():
        arr = np.load(str(fpath))
        print(f"  ✓  verite/{fname} — shape {arr.shape}")
    else:
        print(f"  ✗  {fpath} not found")

NewsCLIPpings CLIP embeddings (ViT-B/32):
  ✓  clip_image_embeddings/test.pkl — 50,000 entries  dim=512
  ✓  clip_text_embeddings/test.pkl — 40,915 entries  dim=512

VERITE CLIP embeddings (ViT-L/14):
  ✓  verite/image_embeddings.npy — shape (1001, 768)
  ✓  verite/text_embeddings.npy — shape (1001, 768)


## 7. Project paths

In [15]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("../../").resolve()))
from src.utils.paths import *

checks = {
    "PROJECT ROOT" : ROOT,
    "data/"        : DATA,
    "cache/"       : CACHE,
    "newsclippings/": NEWSCLIP,
    "verite/"      : VERITE,
    "mmfakebench/" : DATA / "mmfakebench",
    "clip_embeddings/": CLIP_EMB,
}
all_ok = True
for label, path in checks.items():
    exists = path.exists()
    if not exists: all_ok = False
    print(f"  {'✓' if exists else '✗'}  {label:<20}  {path}")

print()
print("✓  All paths OK" if all_ok else "✗  Some paths missing — run setup_eve_rag.sh")

  ✓  PROJECT ROOT          F:\GitHub\eve-rag
  ✓  data/                 F:\GitHub\eve-rag\data
  ✓  cache/                F:\GitHub\eve-rag\cache
  ✓  newsclippings/        F:\GitHub\eve-rag\data\newsclippings
  ✓  verite/               F:\GitHub\eve-rag\data\verite
  ✓  mmfakebench/          F:\GitHub\eve-rag\data\mmfakebench
  ✓  clip_embeddings/      F:\GitHub\eve-rag\cache\clip_embeddings

✓  All paths OK


## 8. Summary

In [16]:
print("=" * 55)
print("  Environment check complete.")
print("=" * 55)
print(f"  Device      : {DEVICE}")
print(f"  SpaCy model : {SPACY_MODEL}")
print(f"  Project     : {ROOT}")
print()
print("  Next: notebooks/01_setup/data_download.ipynb")

  Environment check complete.
  Device      : cuda
  SpaCy model : en_core_web_sm
  Project     : F:\GitHub\eve-rag

  Next: notebooks/01_setup/data_download.ipynb
